# milli_tts — train an Indic + English TTS on a Colab T4
Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.
Realtime loss / accuracy / audio graphs appear in your W&B dashboard.

In [ ]:
# 1) Clone + install
!git clone https://github.com/iNavLabsResearch/milli_tts.git
%cd milli_tts
!bash setup_colab.sh

In [ ]:
# 2) Secrets (do NOT commit these). They flow into config.json's ENV: markers.
import os
os.environ['HF_TOKEN']      = 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
os.environ['WANDB_API_KEY'] = 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

In [ ]:
# 3) (optional) tweak config for the T4 — language, batch size, steps
import json
cfg = json.load(open('config.json'))
cfg['huggingface']['dataset_config'] = 'assamese'   # pick your IndicVoices language
cfg['training']['batch_size'] = 4                   # T4 16GB friendly
cfg['training']['grad_accum_steps'] = 8
cfg['training']['max_steps'] = 20000
json.dump(cfg, open('config.json','w'), indent=2, ensure_ascii=False)
print('config updated')

In [ ]:
# 4) Train (graphs stream live to W&B). Resumes automatically if interrupted.
!python train.py

In [ ]:
# 5) Inference — pass a voice_id + text, get a wav back
from milli_tts.bootstrap import bootstrap
from milli_tts.inference import TTSInferenceEngine
from IPython.display import Audio
bootstrap('config.json')
engine = TTSInferenceEngine()
print('voices:', engine.list_voices()[:10])
res = engine.synthesize('সময়মতে ডেলিভাৰী দিয়াৰ বাবে বহুত ভাল লাগিল',
                        voice_id=engine.list_voices()[0])
print(f'first_chunk={res.first_chunk_ms:.1f}ms total={res.latency_ms:.1f}ms RTF={res.real_time_factor:.2f}x')
res.save('outputs/demo.wav')
Audio('outputs/demo.wav')